# Panels with a T-stiffener modeled as multi-domain assemblies

A flat or curved panel with one T-stiffener is modeled with 2D domains for the
skin, the base and the flange of the stiffener, following Castro and Donadon
(2017) [castro2017Multidomain]. A debonding defect of length `defect_a` is
included at the middle of the panel, where the base is not connected to the
skin.

The connections are imposed exactly with the default
`conn_method='null-space'`, see [Exact connections with the null-space method](../multidomain_null_space.rst), or with penalty
stiffnesses with `conn_method='penalty'`.

Linear buckling, with constant stress resultants or with the pre-buckling
stress state from a static analysis, using
`tstiff2d_1stiff_compression`:

In [1]:
import numpy as np
from panels.multidomain import (
        tstiff2d_1stiff_freq,
        tstiff2d_1stiff_compression,
        tstiff2d_1stiff_flutter,
        )

In [2]:
# from tests/multidomain/test_tstiff2d_assembly.py
def test_tstiff2d_1stiff_compression():
    print('Testing assembly function: tstiff2d_1stiff_compression')
    b = 1.
    bb = b/5.
    bf = bb/2.
    ys = b/2.
    assy, eigvals, eigvecs = tstiff2d_1stiff_compression(
        b=b,
        bb=bb,
        bf=bf,
        a=3.,
        ys=ys,
        defect_a=0.6,
        rho=1.3e3,
        plyt=0.125e-3,
        laminaprop=(142.5e9, 8.7e9, 0.28, 5.1e9, 5.1e9, 5.1e9),
        stack_skin=[0, 45, -45, 90, -45, 45, 0],
        stack_base=[0, 90, 0]*4,
        stack_flange=[0, 90, 0]*8,
        m=8, n=7,
        mb=7, nb=6,
        mf=8, nf=6,
        run_static_case=False,
        Nxx_skin=-1.,
        Nxx_base=-1.,
        Nxx_flange=-1.,
        num_eigvalues=10, conn_method='penalty'
        )
    assert np.isclose(eigvals[0], 142.68, rtol=0.001)

    assy, c, eigvals, eigvecs = tstiff2d_1stiff_compression(
        b=b,
        bb=bb,
        bf=bf,
        a=3.,
        ys=ys,
        defect_a=0.6,
        rho=1.3e3,
        plyt=0.125e-3,
        laminaprop=(142.5e9, 8.7e9, 0.28, 5.1e9, 5.1e9, 5.1e9),
        stack_skin=[0, 45, -45, 90, -45, 45, 0],
        stack_base=[0, 90, 0]*4,
        stack_flange=[0, 90, 0]*8,
        m=8, n=7,
        mb=7, nb=6,
        mf=8, nf=6,
        run_static_case=True,
        Nxx_skin=-1.,
        Nxx_base=-1.,
        Nxx_flange=-1.,
        num_eigvalues=10, conn_method='penalty'
        )
    assert np.isclose(eigvals[0], 114.86, rtol=0.001)

In [3]:
test_tstiff2d_1stiff_compression()
print('test_tstiff2d_1stiff_compression: passed')

Testing assembly function: tstiff2d_1stiff_compression


test_tstiff2d_1stiff_compression: passed


Frequency analysis, using `tstiff2d_1stiff_freq`:

In [4]:
# from tests/multidomain/test_tstiff2d_assembly.py
def test_tstiff2d_1stiff_freq():
    print('Testing assembly function: tstiff2d_1stiff_freq')
    b = 1.
    bb = b/5.
    bf = bb/2.
    ys = b/2.
    assy, eigvals, eigvecs = tstiff2d_1stiff_freq(
        b=b,
        bb=bb,
        bf=bf,
        a=3.,
        ys=ys,
        defect_a=0.1,
        rho=1.3e3,
        plyt=0.125e-3,
        laminaprop=(142.5e9, 8.7e9, 0.28, 5.1e9, 5.1e9, 5.1e9),
        stack_skin=[0, 45, -45, 90, -45, 45, 0],
        stack_base=[0, 90, 0]*4,
        stack_flange=[0, 90, 0]*8,
        m=6, n=7,
        mb=5, nb=6,
        mf=6, nf=7,
        num_eigvalues=10, conn_method='penalty'
        )
    omegan = (-eigvals[0])**0.5
    #NOTE the reference values of this module were changed in 2025 to match
    #      the skin-base connection 'SB' with a hard-coded penalty
    #      kt = 2e5, which is a N/mm**3 value and about 6e-7 of the
    #      penalty given by calc_kt_kr() in these SI units, so the base was
    #      practically disconnected from the skin. Once the default penalty
    #      was restored, the values of 2024 came back
    assert np.isclose(omegan, 48.2733, atol=0.001, rtol=0.001)

In [5]:
test_tstiff2d_1stiff_freq()
print('test_tstiff2d_1stiff_freq: passed')

Testing assembly function: tstiff2d_1stiff_freq


test_tstiff2d_1stiff_freq: passed


Flutter analysis with the piston theory, using
`tstiff2d_1stiff_flutter`:

In [6]:
# from tests/multidomain/test_tstiff2d_assembly.py
def test_tstiff2d_1stiff_flutter():
    print('Testing assembly function: tstiff2d_1stiff_flutter')
    b = 1.
    bb = b/5.
    bf = bb/2.
    ys = b/2.
    assy, eigvals, eigvecs = tstiff2d_1stiff_flutter(
        b=b,
        bb=bb,
        bf=bf,
        a=3.,
        ys=ys,
        defect_a=0.1,
        rho=1.3e3,
        plyt=0.125e-3,
        laminaprop=(142.5e9, 8.7e9, 0.28, 5.1e9, 5.1e9, 5.1e9),
        stack_skin=[0, 45, -45, 90, -45, 45, 0],
        stack_base=[0, 90, 0]*4,
        stack_flange=[0, 90, 0]*8,
        m=6, n=7,
        mb=5, nb=6,
        mf=6, nf=7,
        air_speed=2*343.,
        rho_air=1.2,
        Mach=2,
        speed_sound=343.,
        run_static_case=False,
        num_eigvalues=10, conn_method='penalty'
        )
    omegan = (-eigvals[0])**0.5
    print(eigvals)
    assert np.isclose(omegan, 446.12+0.j, rtol=0.001)

In [7]:
test_tstiff2d_1stiff_flutter()
print('test_tstiff2d_1stiff_flutter: passed')

Testing assembly function: tstiff2d_1stiff_flutter


[-199018.85517227     -0.j         -231513.18155039+141285.19447406j
 -231513.18155039-141285.19447406j -238221.51260813+276499.68333308j
 -238849.23075155+275251.96178846j -238849.23075155-275251.96178846j
 -238221.51260813-276499.68333308j -359447.08582784     -0.j
 -356074.32649713+274514.42557475j -356074.32649713-274514.42557475j]
test_tstiff2d_1stiff_flutter: passed


## References

- **[castro2017Multidomain]** S.G.P. Castro and M.V. Donadon, "Assembly of Semi-Analytical models to Address Linear Buckling and Vibration of Stiffened Composite Panels with Debonding Defect". Composite Structures, 160:232-247, 2017. https://doi.org/10.1016/j.compstruct.2016.10.026.